In [1]:
from gitsource import GithubRepositoryDataReader

reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)

files = reader.read()

Q1. How many lesson pages


In [2]:
documents = []

for file in files:
    doc = file.parse()
    documents.append(doc)

print(f' lesson pages number { len(documents)}')


 lesson pages number 72


Q2. Indexing and searching


In [3]:
# creating the index
from minsearch import Index

def build_index(documents):
    index = Index(
        text_fields=["content"],
        keyword_fields=["filename"]
    )
    index.fit(documents)
    return index


In [4]:
index = build_index(documents)

# searching it

question = "How does the agentic loop keep calling the model until it stops?"

search_results = index.search(
    question,
    num_results=1
)

search_results




[{'content': '# The Agentic Loop\n\nVideo: [Watch this lesson](https://www.youtube.com/watch?v=ePlQUcTPPjw&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)\n\nIn the previous lesson, we did function calling by hand. We sent a\nmessage and got back a function call. We ran it, sent the result back,\nand got the answer.\n\nThat works for one function call. It breaks down when the model wants\nto search several times, or when the first search misses the answer.\nWe don\'t know in advance how many calls the model will want. So we\nneed a loop that keeps calling the model and running tools until it\'s\ndone. An agent is exactly that.\n\n## Anatomy of an agent\n\nWith the LLM in the driver\'s seat, we have an agent. It\'s an AI\nassistant whose goal is to help the user.\n\nAn agent has three parts:\n\n- Instructions, the role and behavior we want. We pass this as the\n  `developer` message. The better the instructions, the better the\n  agent helps.\n- Tools, the functions the agent can call to carry

Q3. RAG


In [5]:
from rag_helper_2 import RAGBase
from openai import OpenAI

openai_client = OpenAI()

assistant = RAGBase(
    index=index,
    llm_client=openai_client,
)

result = assistant.rag(
    "How does the agentic loop keep calling the model until it stops?"
)

print(result["answer"])
print("Input tokens:", result["input_tokens"])

It keeps calling the model in a `while True` loop, and after each response it checks whether there were any `function_call` items.

- If there was a function call, it runs the tool, appends the tool output to the message history, and loops again.
- If there were no function calls on that turn, it breaks out of the loop.

So the stop condition is simply: **no function calls in the latest response**.
Input tokens: 7126


Q4. Chunking


In [6]:
from gitsource import chunk_documents

chunks = chunk_documents(documents, size=2000, step=1000)

print(len(chunks))

295


Q5. RAG with chunking


In [7]:
reindex = build_index(chunks)

assistant = RAGBase(
    index=reindex,
    llm_client=openai_client,
)

result = assistant.rag(
    "How does the agentic loop keep calling the model until it stops?"
)

print(result["answer"])
print("Input tokens:", result["input_tokens"])



The loop keeps calling the model inside a `while True` loop, and after each response it checks whether there were any `function_call` items.

- If the model returns a function call, the code runs the tool, appends the result to the message history, and continues looping.
- If the model returns a message with no function calls, `has_function_calls` stays `False`, and the loop breaks.

So the stopping rule is: **no function calls in the current turn means the agent is done**.
Input tokens: 2309


How many fewer input tokens does the chunked version send?

In [8]:
print(7126/2309)

3.086184495452577


Q6. Turning it into an agent

In [9]:
from toyaikit.llm import OpenAIClient
from toyaikit.tools import Tools
from toyaikit.chat import IPythonChatInterface
from toyaikit.chat.runners import OpenAIResponsesRunner, DisplayingRunnerCallback

In [10]:
search_calls = 0

def search_course(query: str) -> str:
    """
    Search the course materials.
    """
    global search_calls
    search_calls += 1

    results = index.search(query, num_results=5)

    return "\n\n".join(
        doc["content"] for doc in results
    )    

In [11]:
agent_tools = Tools()
agent_tools.add_tool(search_course)

In [12]:
agent_tools.get_tools()

[{'type': 'function',
  'name': 'search_course',
  'description': 'Search the course materials.',
  'parameters': {'type': 'object',
   'properties': {'query': {'type': 'string',
     'description': 'query parameter'}},
   'required': ['query'],
   'additionalProperties': False}}]

Create the chat interface and a callback, then build the runner:



In [13]:
instructions= """
You're a course teaching assistant.
Answer the student's question using the search tool.
Make multiple searches with different keywords before answering.
"""

chat_interface = IPythonChatInterface()
callback = DisplayingRunnerCallback(chat_interface)

runner = OpenAIResponsesRunner(
    tools=agent_tools,
    developer_prompt=instructions,
    chat_interface=chat_interface,
    llm_client=OpenAIClient(model="gpt-5.4-mini")
)

In [14]:
result = runner.loop(
    prompt="How does the agentic loop work, and how is it different from plain RAG?",
    callback=callback,
)


-> Response received


-> Response received
